# m2_custom_models

Member 2 workbench. Here we try ideas for Model A and Model B and test them.
Working code moves later into `src/edgecnn/models/custom/`.
Nothing here produces a report result.

In [32]:
# Check that Python and PyTorch work in this kernel.
import sys

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

Python: 3.14.6
PyTorch: 2.14.1+cpu


## Step 2, topic 1: the Conv layer

Run the cells below one by one, from top to bottom.

- **Cell 1** makes a fake image and passes it through one Conv layer. Look at the shapes.
- **Cell 2** counts the parameters by hand and checks the answer with code.
- **Cell 3** changes the kernel size (3, 5, 7). Look at what changes and what stays the same.

Shape order is `(batch, channels, height, width)`. Batch is how many pictures we send at once.

In [33]:
from torch import nn

torch.manual_seed(0)  # same random numbers every time we run

# A fake image: 1 picture, 3 colour channels (R, G, B), 64 x 64 pixels.
image = torch.randn(1, 3, 64, 64)
print("input shape :", tuple(image.shape))

# One Conv layer: 32 filters, each looks at a 3x3 window of all 3 colour channels.
# padding=1 adds a border of zeros, so the output keeps the same height and width.
conv = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)

out = conv(image)
print("output shape:", tuple(out.shape))

input shape : (1, 3, 64, 64)
output shape: (1, 32, 64, 64)


In [34]:
# Count the parameters by hand, then check with code.
by_hand_weights = 3 * 3 * 3 * 32   # kernel height x kernel width x input channels x filters
by_hand_bias = 32                  # one bias number per filter
print("by hand :", by_hand_weights, "+", by_hand_bias, "=", by_hand_weights + by_hand_bias)

in_code = sum(p.numel() for p in conv.parameters())
print("in code :", in_code)
print("weight tensor shape:", tuple(conv.weight.shape))   # (filters, input channels, height, width)

by hand : 864 + 32 = 896
in code : 896
weight tensor shape: (32, 3, 3, 3)


In [35]:
# Try three kernel sizes. Everything else stays the same.
# Question: what changes? And what does NOT change?
for k in (3, 5, 7):
    layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=k, padding=k // 2)
    out = layer(image)
    n_params = sum(p.numel() for p in layer.parameters())
    print(f"kernel {k}x{k}: output {tuple(out.shape)}, parameters {n_params}")

kernel 3x3: output (1, 32, 64, 64), parameters 896
kernel 5x5: output (1, 32, 64, 64), parameters 2432
kernel 7x7: output (1, 32, 64, 64), parameters 4736


## Step 2, topic 2: max pooling

Pooling makes the picture smaller. **Max pooling** looks at small blocks (here 2 x 2) and keeps only the biggest number in each block.

**Note:** these cells use `torch`, `nn`, `image` and `conv` from the cells above. After a restart, use **Run All** first, or run the cells from the top.

- **Cell 4** shows it on a tiny 4 x 4 example. You can check the answer by hand.
- **Cell 5** applies it to the output of our Conv layer. Look at the shapes and the parameter count.
- **Cell 6** follows the picture size through three pooling steps.

In [36]:
# A tiny 4 x 4 "picture" with one channel. Pooling window: 2 x 2.
x = torch.tensor([[1., 3., 2., 0.],
                  [4., 2., 1., 1.],
                  [0., 1., 5., 2.],
                  [2., 3., 1., 4.]]).reshape(1, 1, 4, 4)

pool = nn.MaxPool2d(kernel_size=2)   # stride is 2 too, so the blocks do not overlap

print(x[0, 0])
print(pool(x)[0, 0])
# Check by hand: the top-left block is [[1, 3], [4, 2]]. Its biggest number is 4.

tensor([[1., 3., 2., 0.],
        [4., 2., 1., 1.],
        [0., 1., 5., 2.],
        [2., 3., 1., 4.]])
tensor([[4., 2.],
        [3., 5.]])


In [37]:
# Max pooling on the output of our Conv layer.
conv_out = conv(image)
pooled = pool(conv_out)

print("before pooling:", tuple(conv_out.shape))
print("after pooling :", tuple(pooled.shape))
print("parameters in the pooling layer:", sum(p.numel() for p in pool.parameters()))

before pooling: (1, 32, 64, 64)
after pooling : (1, 32, 32, 32)
parameters in the pooling layer: 0


In [38]:
# Follow the picture size through three Conv + pooling steps.
# Each pooling step halves the height and the width.
size = 64
for step in range(1, 4):
    size = size // 2
    print(f"after pooling {step}: {size} x {size}")

# How many numbers does the layer output hold, before and after one pooling?
print("before pooling:", 32 * 64 * 64)
print("after pooling :", 32 * 32 * 32)

after pooling 1: 32 x 32
after pooling 2: 16 x 16
after pooling 3: 8 x 8
before pooling: 131072
after pooling : 32768


## Step 2, topic 3: BatchNorm

BatchNorm comes right after Conv. It does **not** change the shape. It keeps the numbers of each channel in a steady range, so training is faster and more stable.

**Note:** these cells use `torch`, `nn` and `conv` from the cells above. Use **Run All** if you restarted.

- **Cell 7** sends 8 uneven pictures through Conv and then BatchNorm. Look at the average and the spread of each channel, before and after.
- **Cell 8** shows what BatchNorm learns (parameters) and what it only remembers (not parameters).
- **Cell 9** counts parameters with and without the Conv bias.

In [39]:
# Make a batch of 8 pictures with uneven numbers (big and shifted), to see what BatchNorm does.
batch = torch.randn(8, 3, 64, 64) * 5 + 3
conv_out = conv(batch)

bn = nn.BatchNorm2d(32)      # 32 = number of channels it has to keep steady
bn_out = bn(conv_out)

def per_channel(t):
    # average and spread of the first 4 channels, over the whole batch
    return t.mean(dim=(0, 2, 3))[:4].detach().numpy().round(2), t.std(dim=(0, 2, 3))[:4].detach().numpy().round(2)

mean_before, std_before = per_channel(conv_out)
mean_after, std_after = per_channel(bn_out)

print("shape before:", tuple(conv_out.shape), " shape after:", tuple(bn_out.shape))
print("before BatchNorm -> average:", mean_before, " spread:", std_before)
print("after  BatchNorm -> average:", mean_after, " spread:", std_after)

shape before: (8, 32, 64, 64)  shape after: (8, 32, 64, 64)
before BatchNorm -> average: [ 1.08 -0.25  1.4   2.18]  spread: [2.96 3.13 2.61 3.27]
after  BatchNorm -> average: [ 0.  0. -0. -0.]  spread: [1. 1. 1. 1.]


In [40]:
# What does BatchNorm learn, and what does it only remember?
print("Learned (these are parameters):")
for name, p in bn.named_parameters():
    print(" ", name, tuple(p.shape), p.numel())

print("Remembered (these are NOT parameters):")
for name, b in bn.named_buffers():
    print(" ", name, tuple(b.shape))

print("BatchNorm parameters in total:", sum(p.numel() for p in bn.parameters()))

Learned (these are parameters):
  weight (32,) 32
  bias (32,) 32
Remembered (these are NOT parameters):
  running_mean (32,)
  running_var (32,)
  num_batches_tracked ()
BatchNorm parameters in total: 64


In [41]:
# Conv with a bias, and Conv without a bias followed by BatchNorm.
def count(layer):
    return sum(p.numel() for p in layer.parameters())

conv_with_bias = nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True)
conv_no_bias = nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False)

print("Conv with bias            :", count(conv_with_bias))
print("Conv without bias         :", count(conv_no_bias))
print("Conv without bias + BN    :", count(conv_no_bias) + count(bn))
print("Conv with bias + BN       :", count(conv_with_bias) + count(bn))
# BatchNorm already shifts every channel, so the Conv bias does nothing useful.

Conv with bias            : 896
Conv without bias         : 864
Conv without bias + BN    : 928
Conv with bias + BN       : 960


## Step 2, topic 4: ReLU (the activation)

ReLU comes after BatchNorm. The rule is very simple: **if the number is negative, make it 0. Otherwise keep it.** In short, `max(0, x)`.

**Note:** these cells use `torch`, `nn` and `bn_out` from the cells above. Use **Run All** if you restarted.

- **Cell 10** applies ReLU and ReLU6 to a few numbers. Check the answer by hand first.
- **Cell 11** shows why we need an activation at all. Without it, many layers behave like one layer.
- **Cell 12** applies ReLU to the BatchNorm output. Look at the shape and how many numbers became 0.

The assignment asks for a **hardware-aware reason** for the activation. We will talk about that after you see how simple ReLU is.

In [42]:
# ReLU and ReLU6 on a few numbers. Check by hand first.
values = torch.tensor([-3., -1., 0., 2., 5., 8.])

print("input :", values)
print("ReLU  :", nn.ReLU()(values))      # max(0, x)
print("ReLU6 :", nn.ReLU6()(values))     # min(max(0, x), 6)

input : tensor([-3., -1.,  0.,  2.,  5.,  8.])
ReLU  : tensor([0., 0., 0., 2., 5., 8.])
ReLU6 : tensor([0., 0., 0., 2., 5., 6.])


In [43]:
# Why do we need an activation at all?
# Two layers WITHOUT an activation between them are the same as ONE layer.
torch.manual_seed(1)
layer1 = nn.Linear(4, 3)
layer2 = nn.Linear(3, 2)
v = torch.randn(5, 4)

two_layers = layer2(layer1(v))

# Build one single layer that does the same work.
one_layer = nn.Linear(4, 2)
with torch.no_grad():
    one_layer.weight.copy_(layer2.weight @ layer1.weight)
    one_layer.bias.copy_(layer2.weight @ layer1.bias + layer2.bias)

print("Same result from one layer?", torch.allclose(two_layers, one_layer(v), atol=1e-6))

# With a ReLU in the middle, this trick no longer works.
with_relu = layer2(torch.relu(layer1(v)))
print("Same result with ReLU in the middle?", torch.allclose(with_relu, one_layer(v), atol=1e-6))

Same result from one layer? True
Same result with ReLU in the middle? False


In [44]:
# ReLU on the output of BatchNorm (from cell 7).
relu = nn.ReLU()
act_out = relu(bn_out)

print("shape before:", tuple(bn_out.shape), " shape after:", tuple(act_out.shape))
print("smallest value before:", round(bn_out.min().item(), 2), " after:", round(act_out.min().item(), 2))
print("share of numbers that are exactly 0 before:", round((bn_out == 0).float().mean().item(), 3))
print("share of numbers that are exactly 0 after :", round((act_out == 0).float().mean().item(), 3))
print("parameters in ReLU:", sum(p.numel() for p in relu.parameters()))

shape before: (8, 32, 64, 64)  shape after: (8, 32, 64, 64)
smallest value before: -4.73  after: 0.0
share of numbers that are exactly 0 before: 0.0
share of numbers that are exactly 0 after : 0.5
parameters in ReLU: 0
